# API Latency

Thirty minutes of traffic to a small web API: one row per request, in `data/requests.csv`.
Let's find out how fast it is, which endpoint is slowest, and what happened at 10:18.

This notebook runs on the booth's **Python 3.13 (conda)** kernel, where conda installed pandas and
matplotlib. *Run → Run All Cells* to fill in every output.

## 1. Load the log

`src/latency.py` reads the CSV with pandas; `just run` prints a report from the same function.

In [ ]:
import sys
sys.path.insert(0, "src")

import pandas as pd
from latency import load, report

log = load()
log.head()

In [ ]:
print(f"{len(log)} requests from {log['timestamp'].min():%H:%M} to {log['timestamp'].max():%H:%M}")
log["duration_ms"].describe().round(1)

## 2. Percentiles per endpoint

The average hides the slow requests users actually notice, so look at p95 and p99: 95% (or 99%)
of requests were faster than this.

In [ ]:
report(log).round(1)

## 3. What does "slow" look like?

Most requests are quick, with a long tail to the right. The dashed lines mark p50 and p95.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log["duration_ms"], bins=60, range=(0, 2000), color="tab:blue", alpha=0.8)
for q, label in [(0.50, "p50"), (0.95, "p95")]:
    value = log["duration_ms"].quantile(q)
    ax.axvline(value, color="black", linestyle="--", linewidth=1)
    ax.annotate(f"{label} {value:.0f} ms", (value, ax.get_ylim()[1] * 0.9), xytext=(4, 0), textcoords="offset points")
ax.set_xlabel("Request duration (ms)")
ax.set_ylabel("Requests")
ax.set_title("Latency distribution (requests over 2 s not shown)")
plt.show()

In [ ]:
p95 = report(log)["p95_ms"].sort_values()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(p95.index, p95.values, color="tab:orange")
ax.set_xlabel("p95 latency (ms)")
ax.set_title("p95 latency by endpoint")
plt.show()

## 4. What happened at 10:18?

Group the requests into one-minute buckets and count the 5xx errors. Something broke for about
four minutes; the second chart shows which endpoint it was.

In [ ]:
per_minute = log.set_index("timestamp").resample("1min")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
per_minute["is_error"].sum().plot(ax=top, color="tab:red", marker="o")
top.set_ylabel("5xx errors / min")
top.set_title("Errors and latency over time")

search = log[log["endpoint"] == "/api/search"].set_index("timestamp")
search["duration_ms"].resample("1min").quantile(0.95).plot(ax=bottom, color="tab:purple", marker="o", label="/api/search")
log.set_index("timestamp")["duration_ms"].resample("1min").quantile(0.95).plot(ax=bottom, color="gray", label="all endpoints")
bottom.set_ylabel("p95 latency (ms)")
bottom.set_xlabel("Time")
bottom.legend()
plt.show()

In [ ]:
incident = log[(log["timestamp"].dt.strftime("%H:%M") >= "10:18") & (log["timestamp"].dt.strftime("%H:%M") < "10:22")]
report(incident).round(1)

## 5. Your turn

Pick an endpoint, or change the percentile. Need another package? `conda install <pkg>` in a
terminal, then *Kernel → Restart Kernel*.

In [ ]:
endpoint = "/api/orders"   # try "/api/users" or "/health"

subset = log[log["endpoint"] == endpoint]
subset.groupby("status").size().rename("requests")